# ingest 诊断：`run_folder.ipynb` 慢在哪

## 目的

用**实测数据**回答"`notebooks/run_folder.ipynb` 的 ingest 慢在哪"：把一份文档的 ingest 拆成 pdfspine 抽取、ONNX 版面、回退页 LLM、embedding、文件系统五个耗时项，各自实测单价、外推到整份文档，再汇成一张排序表。需要用到的数字若只能假设（每页向量数、每页文件数、回退页比例），表里会明确标注"假设"，并用第 9 节的线程栈采样去验证。

## 怎么用

1. 把本 notebook 放在 `notebooks/` 下（和 `run_folder.ipynb` 同目录），**Run All**，约 3～5 分钟（文件系统基准在 FUSE 上较慢；第 9 节固定要等约 30 秒）。
2. **可以和正在跑的 `run_folder` 同时运行**：本 notebook 只读 ingestion 目录，写盘只发生在 `ROOT_DIR/data/_diagnostics/`、`/tmp/ragspine-diag/`、`/local_disk0/ragspine-diag/` 三处，基准文件会在第 4 节结束后清理。注意：同时运行会和正在跑的 ingest 争抢 CPU / 文件系统，实测值会偏大，这本身也是真实环境的一部分。
3. 环境变量和 `run_folder` 相同（`.env` 里的 `NB_PDF_DIR`、`APP_ONNX_LAYOUT_MODEL`、LLM / embedding 配置）。未配置的项只会打印 `UNAVAILABLE: <原因>` 然后继续。
4. 第 8 节默认会产生 **1～2 次计费的 LLM 调用**；不想调用就把该节开头的 `RUN_LLM_PROBE` 改成 `False`（或运行前设环境变量 `DIAG_RUN_LLM_PROBE=0`）。
5. 跑完把最后一节打印的 `ingest_diagnostics_<时间戳>.json` / `.md` 发回即可。

## 每个 section 回答什么

| 节 | 问题 |
|---|---|
| 1 Setup | 路径落在哪种挂载（FUSE / 本地盘）上？ |
| 2 运行环境 | 几个核、多少内存？ONNX 能不能用？`LAYOUT_POLICY="auto"` 实际会选什么？ |
| 3 当前 ingest 的进度 | 已经完成多少份、吞吐是多少？扫一份文档的 `processing/` 要多久（FUSE 延迟的旁证）？ |
| 4 文件系统基准 | 每个文件落位（写 + fsync + 链接 / rename + 读回）多少 ms？`os.link` 是否失败回退？校验重读单价？ |
| 5 pdfspine 抽取基准 | 每页抽取多少 ms？整份文档多少秒？ |
| 6 ONNX 版面基准 | 每页版面多少秒？并发文档间是否争抢线程？ |
| 7 Embedding 延迟 | 一次 16 条批请求多少秒？全局串行锁下总共多久？ |
| 8 LLM 延迟 | 文本 / vision 单次调用多少秒？ |
| 9 线程栈采样 | 正在跑的 ingest 进程，线程此刻卡在哪一类调用上？ |
| 10 汇总判断 | 上面所有数字汇成一张表，谁是第一名？ |

## 第 10 节的表怎么读

- **每份文档估计秒数**：该耗时项处理一份文档的秒数，已按秒数从大到小排序。
- **依据**：`实测` = 直接测到；`外推` = 实测单价 × 页数等实测量；`假设` = 含我们假设的比例（每页 10 个向量、每页 80 个文件、20% 回退页），**这类数字必须用第 9 节栈采样验证**。
- **能否被并发摊薄**：`MAX_PARALLEL_DOCUMENTS` 份文档并行时，这一项的总耗时是否会按并行度下降。"否"的项（如全局串行的 embedding）会成为并发之后的天花板。
- 结论段会给出第一名比第二名大几倍，并与第 9 节栈采样里最多的桶对照：两者一致，结论就比较可靠；不一致，以栈采样为准、回头检查假设。


In [ ]:
# 与 run_folder.ipynb 一致的安装方式（editable + extras；pdf-onnx = onnxruntime，ONNX 版面要用）。只在 Databricks 上执行，其它环境（本地验证）跳过。
# 装完需要重启解释器（下一格）。
import os

if "DATABRICKS_RUNTIME_VERSION" in os.environ:
    get_ipython().run_line_magic("pip", 'install -q -e "..[pdf,pdf-onnx,service]"')  # noqa: F821 - IPython 注入
else:
    print("非 Databricks 环境：跳过 %pip install -e（本地请确保已 pip install -e . ）")

In [ ]:
import os

# 只在 Databricks 上重启 Python（让上一格 %pip 装的包生效）；其它环境什么都不做
if os.environ.get("DATABRICKS_RUNTIME_VERSION") and "dbutils" in globals():
    dbutils.library.restartPython()  # noqa: F821 - dbutils 由 Databricks 注入
else:
    print("非 Databricks 环境，跳过")

## 1. Setup

路径、挂载类型。setup 与 `run_folder.ipynb` 一致，保证在 Databricks 上行为可比。

In [ ]:
print("=" * 70)
print("1. Setup")

# editable 安装靠 .pth 把 <repo>/src 加进 sys.path；Databricks 上该 .pth 不生效时显式放到最前（与 run_folder 一致）
import sys
from pathlib import Path

_src = Path.cwd().parent / "src"
if (_src / "ragspine" / "__init__.py").is_file() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))
print("src 路径 =", _src, "| 已在 sys.path:", str(_src) in sys.path)

import json
import os
import platform
import re
import shutil
import subprocess
import tempfile
import threading
import time
import traceback
import unicodedata
from collections import Counter
from datetime import UTC, datetime
from hashlib import sha256

from enterprise_pdf_rag.adapters.ingest_mode import (
    choose_layout_policy,
    choose_unverified_table_structure,
)
from ragspine.common.evidence.configs import ROOT_DIR
from ragspine.common.evidence.configs import get_settings

settings = get_settings()
PDF_DIR = settings.pdf_source_dir  # NB_PDF_DIR
QUESTIONS = settings.questions_path
REPORT_DIR = ROOT_DIR / "data" / "reports" / ("run-folder" if QUESTIONS is None else QUESTIONS.stem)
INGESTION_ROOT = settings.ingestion_root
DATA_DIR = (ROOT_DIR / "data").resolve()
DIAG_DIR = DATA_DIR / "_diagnostics"

R: dict = {}  # 各 section 的实测结果，最后一节汇总并落盘


def dispw(text) -> int:
    return sum(2 if unicodedata.east_asian_width(ch) in "WF" else 1 for ch in str(text))


def print_table(headers, rows) -> None:
    rows = [[str(c) for c in row] for row in rows]
    widths = [max(dispw(x) for x in col) for col in zip(headers, *rows)] if rows else [dispw(h) for h in headers]
    line = lambda cells: "  ".join(str(c) + " " * (w - dispw(c)) for c, w in zip(cells, widths))  # noqa: E731
    print(line(headers))
    print("  ".join("-" * w for w in widths))
    for row in rows:
        print(line(row))


def section(name: str) -> None:
    print("=" * 70)
    print(name)


def unavailable(reason) -> None:
    print(f"UNAVAILABLE: {reason}")


def err(exc: BaseException) -> str:
    return f"{type(exc).__name__}: {exc}"


def mount_kind(path) -> str:
    s = str(path)
    if s.startswith("/Workspace"):
        return "Workspace files (FUSE)"
    if s.startswith("/dbfs"):
        return "DBFS (FUSE)"
    if s.startswith("/Volumes"):
        return "UC Volume (FUSE)"
    if s.startswith(("/local_disk0", "/tmp", "/private/tmp", "/databricks/driver")):
        return "local disk"
    return "unknown"


def proc_mount(path) -> str:
    try:
        real = os.path.realpath(str(path))
        best = None
        with open("/proc/mounts", encoding="utf-8") as handle:
            for line in handle:
                parts = line.split()
                if len(parts) < 3:
                    continue
                point = parts[1].replace("\\040", " ")
                if real == point or real.startswith(point.rstrip("/") + "/") or point == "/":
                    if best is None or len(point) > len(best[0]):
                        best = (point, parts[2])
        return f"{best[1]} @ {best[0]}" if best else "no match"
    except OSError as exc:
        return f"n/a ({type(exc).__name__})"


_rows = []
for _label, _p in (
    ("ROOT_DIR", ROOT_DIR),
    ("DATA_DIR", DATA_DIR),
    ("INGESTION_ROOT", INGESTION_ROOT),
    ("REPORT 目录", REPORT_DIR),
    ("NB_PDF_DIR", PDF_DIR if PDF_DIR is not None else "(未设置)"),
):
    if _p == "(未设置)":
        _rows.append((_label, _p, "-", "-"))
    else:
        _rows.append((_label, _p, mount_kind(_p), proc_mount(_p)))
print_table(("名称", "路径", "挂载类型", "/proc/mounts fstype"), _rows)
R["paths"] = {r[0]: [str(r[1]), r[2], r[3]] for r in _rows}

In [ ]:
# ONNX 版面自检（ADR 0030 / 0039）：本地 PP-DocLayoutV3 版面能否用、权重在哪。onnxruntime 由安装格的
# pdf-onnx extra 装上；权重不随任何 wheel 分发：已配置 APP_ONNX_LAYOUT_MODEL（Databricks 上指向 Volume 里的目录）
# 但文件缺失时，这里从 pdfspine 记录的 ModelScope 地址下载到该位置。没配置 / 下载失败只打印提示、不中断，
# 此时 LAYOUT_POLICY = "auto" 会退到 "deterministic-text-pages"（配置格会再打印原因）。
from enterprise_pdf_rag.adapters.onnx_partition import (
    ensure_onnx_layout_weights,
    onnx_layout_status,
)
from ragspine.common.evidence.configs import get_settings

print(ensure_onnx_layout_weights(get_settings().onnx_layout_model))
print(onnx_layout_status(get_settings().onnx_layout_model))
# 版面路由拿不准的页交给谁（ADR 0039）：model = 每页一次 VLM 版面（默认）；onnx-accept / text-only = 不调 VLM 版面
print("版面回退策略 APP_LAYOUT_FALLBACK =", get_settings().layout_fallback)

## 2. 运行环境

CPU / 内存 / pdfspine / onnxruntime，以及 `run_folder` 在这台机器上实际会选到的 `LAYOUT_POLICY`。

In [ ]:
section("2. 运行环境")
env_info: dict = {}
try:
    env_info["cpu_count"] = os.cpu_count()
    try:
        env_info["affinity_cpus"] = len(os.sched_getaffinity(0))
    except (AttributeError, OSError) as exc:
        env_info["affinity_cpus"] = f"UNAVAILABLE ({type(exc).__name__})"
    try:
        with open("/proc/meminfo", encoding="utf-8") as handle:
            kb = next(int(line.split()[1]) for line in handle if line.startswith("MemTotal"))
        env_info["mem_total_gb"] = round(kb / 1024 / 1024, 1)
    except (OSError, StopIteration, ValueError) as exc:
        env_info["mem_total_gb"] = f"UNAVAILABLE ({type(exc).__name__})"
    env_info["python"] = sys.version.split()[0]
    env_info["platform"] = platform.platform()
    for _name in ("DATABRICKS_RUNTIME_VERSION", "IS_SERVERLESS", "DB_IS_DRIVER", "DB_INSTANCE_TYPE", "SPARK_LOCAL_DIRS"):
        env_info[_name] = os.environ.get(_name, "(未设置)")
    try:
        import pdfspine

        env_info["pdfspine"] = pdfspine.__version__
    except Exception as exc:  # noqa: BLE001
        env_info["pdfspine"] = f"UNAVAILABLE: {err(exc)}"
    try:
        import onnxruntime as ort

        env_info["onnxruntime"] = ort.__version__
        env_info["ort_providers"] = ort.get_available_providers()
    except Exception as exc:  # noqa: BLE001
        env_info["onnxruntime"] = f"UNAVAILABLE: {err(exc)}"
    env_info["onnx_layout_model"] = settings.onnx_layout_model or "(未设置)"
    env_info["PDFSPINE_ONNX_MODELS"] = os.environ.get("PDFSPINE_ONNX_MODELS", "(未设置)")
    _policy, _reason = choose_layout_policy("auto", ingest_mode="lite", onnx_layout_model=settings.onnx_layout_model)
    _table, _table_reason = choose_unverified_table_structure("auto", ingest_mode="lite")
    env_info["LAYOUT_POLICY(auto)"] = _policy
    env_info["LAYOUT_REASON"] = _reason
    env_info["UNVERIFIED_TABLE_STRUCTURE(auto)"] = _table
    env_info["TABLE_STRUCTURE_REASON"] = _table_reason
    _pyspy = shutil.which("py-spy") or (str(Path(sys.executable).parent / "py-spy") if (Path(sys.executable).parent / "py-spy").exists() else None)
    env_info["py-spy"] = _pyspy or "UNAVAILABLE: 未安装（第 9 节会尝试 pip 安装）"
    print_table(("项目", "值"), [(k, v) for k, v in env_info.items()])
    print("\nrun_folder 在这台机器上实际会选: LAYOUT_POLICY =", _policy, "| UNVERIFIED_TABLE_STRUCTURE =", _table)
except Exception as exc:  # noqa: BLE001
    unavailable(err(exc))
R["env"] = env_info
ONNX_USABLE = env_info.get("LAYOUT_POLICY(auto)") == "onnx-layout"

## 3. 当前 ingest 的进度

只读扫描 ingestion 目录：已完成份数、吞吐、`processing/` 的文件数与扫描耗时。

In [ ]:
section("3. 当前 ingest 的进度")
prog: dict = {}
try:
    _t0 = time.perf_counter()
    root = Path(INGESTION_ROOT)
    if not root.is_dir():
        raise FileNotFoundError(f"INGESTION_ROOT 不存在: {root}")
    docs = [e for e in os.scandir(root) if e.is_dir() and re.fullmatch(r"[0-9a-f]{64}", e.name)]
    print(f"列出 INGESTION_ROOT 用时 {time.perf_counter() - _t0:.2f}s（{len(docs)} 个 sha 目录）")
    done: list[tuple[float, str]] = []
    for e in docs:
        try:
            done.append((os.stat(os.path.join(e.path, "processing", "current-processing")).st_mtime, e.name))
        except OSError:
            continue
    done.sort()
    prog.update(total_dirs=len(docs), published=len(done))
    print(f"sha 子目录总数 : {len(docs)}")
    print(f"已发布(有 processing/current-processing): {len(done)}")
    if done:
        hist = Counter(datetime.fromtimestamp(m, UTC).strftime("%Y-%m-%d %H:00") for m, _ in done)
        peak = max(hist.values())
        print("\n每小时完成份数（UTC，按 current-processing 的 mtime）:")
        for hour in sorted(hist):
            print(f"  {hour}  {'#' * max(1, round(40 * hist[hour] / peak)):<40} {hist[hour]}")
        first, last = done[0][0], done[-1][0]
        fmt = lambda t: datetime.fromtimestamp(t, UTC).strftime("%Y-%m-%d %H:%M:%S UTC")  # noqa: E731
        print(f"\n最早完成: {fmt(first)}\n最新完成: {fmt(last)}")
        prog.update(first=fmt(first), last=fmt(last))
        if len(done) >= 2:
            avg = (last - first) / (len(done) - 1)
            prog["avg_gap_s"] = round(avg, 1)
            print(f"平均每份耗时(相邻完成间隔，是吞吐不是单份延迟；并发 N 份时单份延迟约 N 倍): {avg:.1f}s")
        else:
            print("完成份数不足 2，无法算平均耗时")
        scans = []
        for _mtime, name in done[-3:]:
            t0 = time.perf_counter()
            files = size = entries = 0
            stack = [os.path.join(INGESTION_ROOT, name, "processing")]
            while stack:
                with os.scandir(stack.pop()) as it:
                    for item in it:
                        entries += 1
                        if item.is_dir(follow_symlinks=False):
                            stack.append(item.path)
                        else:
                            files += 1
                            size += item.stat(follow_symlinks=False).st_size
            dt = time.perf_counter() - t0
            scans.append(dict(doc=name[:12], files=files, mb=round(size / 1e6, 2), scan_s=round(dt, 2), ms_per_entry=round(dt * 1000 / max(entries, 1), 3)))
        print("\n已完成文档 processing/ 的递归扫描（最近 3 份；扫描耗时本身是文件系统延迟的证据）:")
        print_table(("doc", "文件数", "MB", "扫描秒", "ms/条目"), [(s["doc"], s["files"], s["mb"], s["scan_s"], s["ms_per_entry"]) for s in scans])
        prog["scans"] = scans
        if scans:
            prog["files_per_doc_measured"] = round(sum(s["files"] for s in scans) / len(scans))
    else:
        print("还没有已发布的文档")
except Exception as exc:  # noqa: BLE001
    unavailable(err(exc))
R["progress"] = prog

## 4. 文件系统基准

三个位置对比：原始 mkstemp/fsync/replace，仓库的 `link_new_file`，以及校验阶段的 stat + 重读。结束后清理。

In [ ]:
section("4. 文件系统基准")
from ragspine.common.evidence import file_placement as _fp

BENCH_N = 200
fs_results: dict = {}


def _bench_one(label: str, base: Path) -> None:
    root = base
    root.mkdir(parents=True, exist_ok=True)
    try:
        n = BENCH_N
        payload = b"x" * 512
        # 预热 5 次，单价过高(>0.3s/文件)就缩小次数，避免 FUSE 上跑太久
        t0 = time.perf_counter()
        for i in range(5):
            fd, tmp = tempfile.mkstemp(dir=root)
            os.write(fd, payload)
            os.fsync(fd)
            os.close(fd)
            os.replace(tmp, root / f"warm-{i}")
        warm = (time.perf_counter() - t0) / 5
        if warm > 0.3:
            n = 40
            print(f"  [{label}] 预热单价 {warm * 1000:.0f} ms/文件，次数缩减为 {n}")
        # 原始模式
        t0 = time.perf_counter()
        for i in range(n):
            fd, tmp = tempfile.mkstemp(dir=root)
            os.write(fd, payload + str(i).encode())
            os.fsync(fd)
            os.close(fd)
            target = root / f"raw-{i}"
            os.replace(tmp, target)
            target.read_bytes()
        raw_ms = (time.perf_counter() - t0) * 1000 / n
        # 仓库模式：mkstemp + 写 + fsync + link_new_file(目标 = 内容寻址名) + 清理 tmp
        fails: list[str] = []
        orig_link = os.link

        def counting_link(src, dst, *a, **k):
            try:
                return orig_link(src, dst, *a, **k)
            except OSError as exc:
                fails.append(f"errno={exc.errno}")
                raise

        placed: list[Path] = []
        os.link = counting_link
        try:
            with _fp.recording_repairs() as repairs:
                t0 = time.perf_counter()
                for i in range(n):
                    body = payload + f"repo-{i}".encode()
                    fd, tmp = tempfile.mkstemp(dir=root)
                    os.write(fd, body)
                    os.fsync(fd)
                    os.close(fd)
                    target = root / sha256(body).hexdigest()
                    _fp.link_new_file(Path(tmp), target)
                    Path(tmp).unlink(missing_ok=True)
                    placed.append(target)
                repo_ms = (time.perf_counter() - t0) * 1000 / n
        finally:
            os.link = orig_link
        # 校验重读：stat + read_bytes
        t0 = time.perf_counter()
        for i in range(n):
            os.stat(placed[i % len(placed)])
        stat_ms = (time.perf_counter() - t0) * 1000 / n
        t0 = time.perf_counter()
        for i in range(n):
            placed[i % len(placed)].read_bytes()
        read_ms = (time.perf_counter() - t0) * 1000 / n
        fs_results[label] = dict(
            n=n,
            raw_ms=round(raw_ms, 3),
            repo_ms=round(repo_ms, 3),
            stat_ms=round(stat_ms, 3),
            read_ms=round(read_ms, 3),
            link_failures=len(fails),
            link_fail_detail=sorted(set(fails)),
            repairs=dict(repairs),
        )
    finally:
        shutil.rmtree(root, ignore_errors=True)
        parent = root.parent
        if parent.name == "ragspine-diag":
            try:
                parent.rmdir()
            except OSError:
                pass


for _label, _base in (
    ("ROOT_DIR/data", DIAG_DIR / "fsbench"),
    ("/tmp", Path("/tmp/ragspine-diag/fsbench")),
    ("/local_disk0", Path("/local_disk0/ragspine-diag/fsbench")),
):
    try:
        if _label == "/local_disk0" and not Path("/local_disk0").is_dir():
            print(f"[{_label}] 跳过：/local_disk0 不存在（serverless 或非 Databricks）")
            continue
        print(f"[{_label}] 测试中 … ({_base})")
        _bench_one(_label, _base)
    except Exception as exc:  # noqa: BLE001
        print(f"[{_label}] UNAVAILABLE: {err(exc)}")
        shutil.rmtree(_base, ignore_errors=True)

if fs_results:
    print()
    print_table(
        ("位置", "次数", "原始 ms/文件", "仓库 ms/文件", "stat ms/次", "read ms/次", "os.link 失败次数", "repair 计数"),
        [
            (k, v["n"], v["raw_ms"], v["repo_ms"], v["stat_ms"], v["read_ms"], f'{v["link_failures"]} {v["link_fail_detail"] or ""}', v["repairs"] or "{}")
            for k, v in fs_results.items()
        ],
    )
    print("\n说明: 原始 = mkstemp+write+fsync+replace+read；仓库 = 同样的 tmp 写入后用 link_new_file 落位(os.link 失败则回退 rename+读回)。")
    print("os.link 失败次数 > 0 表示仓库走了回退路径(每个文件多一次 exists + read + replace + read 往返)。")
else:
    unavailable("没有任何位置完成基准")
R["fsbench"] = fs_results

## 5. pdfspine 抽取基准

`NB_PDF_DIR` 第一份 PDF，前 20 页的 `get_text("dict")` + `get_svg_image(text_as_path=False)`。

In [ ]:
section("5. pdfspine 抽取基准")
pdfb: dict = {}
SAMPLE_PDF = None
try:
    if PDF_DIR is None:
        raise RuntimeError("NB_PDF_DIR 未设置")
    _all_pdfs = sorted(Path(PDF_DIR).expanduser().rglob("*.pdf"))
    R["pdf_count"] = len(_all_pdfs)
    if not _all_pdfs:
        raise RuntimeError(f"{PDF_DIR} 下没有 PDF")
    SAMPLE_PDF = _all_pdfs[0]
    from enterprise_pdf_rag.adapters import pdfspine_document as _pd
    from enterprise_pdf_rag.adapters.pdf_password import open_pdf

    t0 = time.perf_counter()
    data = SAMPLE_PDF.read_bytes()
    read_s = time.perf_counter() - t0
    t0 = time.perf_counter()
    doc = open_pdf(data)
    open_s = time.perf_counter() - t0
    pages = doc.page_count
    k = min(20, pages)
    t_text, t_svg, t_full = [], [], []
    digest = sha256(data).hexdigest()
    for i in range(k):
        page = doc.load_page(i)
        t0 = time.perf_counter()
        page.get_text("dict")
        t_text.append(time.perf_counter() - t0)
        t0 = time.perf_counter()
        page.get_svg_image(text_as_path=False)
        t_svg.append(time.perf_counter() - t0)
        try:
            t0 = time.perf_counter()
            _pd._extract_page(page, source_digest=digest, page_index=i)  # 生产路径整页(含校验 / 解析)
            t_full.append(time.perf_counter() - t0)
        except Exception as exc:  # noqa: BLE001
            t_full = []
            pdfb["full_path_error"] = err(exc)
            break
    doc.close()
    ms = lambda xs: sum(xs) * 1000 / len(xs)  # noqa: E731
    pdfb.update(
        pdf=SAMPLE_PDF.name,
        bytes=len(data),
        pages=pages,
        sampled=k,
        read_s=round(read_s, 3),
        open_s=round(open_s, 3),
        text_ms=round(ms(t_text), 2),
        svg_ms=round(ms(t_svg), 2),
    )
    base_ms = pdfb["text_ms"] + pdfb["svg_ms"]
    if t_full:
        pdfb["full_ms"] = round(ms(t_full), 2)
        per_page = pdfb["full_ms"]
    else:
        per_page = base_ms
    pdfb["ms_per_page"] = per_page
    pdfb["doc_seconds"] = round(per_page * pages / 1000 + read_s + open_s, 2)
    print_table(
        ("PDF", "页数", "大小KB", "读文件s", "open s", "采样页", "get_text ms/页", "get_svg ms/页", "生产路径整页 ms/页"),
        [(SAMPLE_PDF.name, pages, len(data) // 1024, pdfb["read_s"], pdfb["open_s"], k, pdfb["text_ms"], pdfb["svg_ms"], pdfb.get("full_ms", "UNAVAILABLE"))],
    )
    print(f"\n外推整份文档: {per_page} ms/页 × {pages} 页 + 读/open = {pdfb['doc_seconds']} s")
    if "full_path_error" in pdfb:
        print("生产路径整页计时失败:", pdfb["full_path_error"])
except Exception as exc:  # noqa: BLE001
    unavailable(err(exc))
R["pdfspine"] = pdfb

## 6. ONNX 版面基准

前 10 页的 `find_layout`，以及并发争抢探测。

In [ ]:
section("6. ONNX 版面基准")
onx: dict = {}
try:
    if not ONNX_USABLE:
        print(f"ONNX 不可用: {env_info.get('LAYOUT_REASON', '未知')}")
        print("=> run_folder 在这台机器上会走 deterministic-text-pages(含图页回退模型版面)，本节无 ONNX 耗时。")
        onx["usable"] = False
    elif SAMPLE_PDF is None:
        raise RuntimeError("第 5 节没有取到样例 PDF")
    else:
        from concurrent.futures import ThreadPoolExecutor

        import onnxruntime as ort

        from enterprise_pdf_rag.adapters.onnx_partition import (
            ONNX_RENDER_DPI,
            ONNX_SUSPECT_VISUAL_SCORE,
            resolve_onnx_layout_model,
        )
        from enterprise_pdf_rag.adapters.pdf_password import open_pdf

        onx["usable"] = True
        model_path = resolve_onnx_layout_model(settings.onnx_layout_model)
        # 与 make_onnx_page_partitioner 同样的选项；find_layout 以关键字参数接收
        options = {"layout_model": os.fspath(model_path), "layout_threshold": ONNX_SUSPECT_VISUAL_SCORE, "dpi": ONNX_RENDER_DPI}
        data = SAMPLE_PDF.read_bytes()
        doc = open_pdf(data)
        n_pages = doc.page_count
        k = min(10, n_pages)
        times = []
        for i in range(k):
            t0 = time.perf_counter()
            doc.load_page(i).find_layout(**options)
            times.append(time.perf_counter() - t0)
        doc.close()
        cold = times[0]
        warm = times[1:] or times
        s_page = sum(warm) / len(warm)
        onx.update(model=str(model_path), pages_total=n_pages, sampled=k, first_page_s=round(cold, 2), warm_s_per_page=round(s_page, 2))
        onx["doc_minutes"] = round(s_page * n_pages / 60, 2)
        print_table(
            ("采样页", "首页 s(含会话加载)", "之后 s/页", "整份文档分钟(外推)"),
            [(k, onx["first_page_s"], onx["warm_s_per_page"], onx["doc_minutes"])],
        )
        # 并发争抢探测：同样几页，串行(上面已测) vs 4 线程各开一个文档同时跑
        try:
            idx = list(range(1, min(5, k)))
            if len(idx) >= 2:

                def _one(i):
                    d = open_pdf(data)
                    try:
                        t = time.perf_counter()
                        d.load_page(i).find_layout(**options)
                        return time.perf_counter() - t
                    finally:
                        d.close()

                t0 = time.perf_counter()
                with ThreadPoolExecutor(len(idx)) as pool:
                    list(pool.map(_one, idx))
                wall = time.perf_counter() - t0
                seq = sum(times[i] for i in idx)
                onx["concurrency"] = dict(threads=len(idx), sequential_s=round(seq, 2), concurrent_wall_s=round(wall, 2), speedup=round(seq / wall, 2))
                print(f"\n并发探测: {len(idx)} 页串行合计 {seq:.2f}s, {len(idx)} 线程并发墙钟 {wall:.2f}s => 加速比 {seq / wall:.2f}x (理想 {len(idx)}x；接近 1x 说明线程间在争抢 CPU / GIL)")
        except Exception as exc:  # noqa: BLE001
            print("并发探测 UNAVAILABLE:", err(exc))
        so = ort.SessionOptions()
        onx["intra_op_num_threads_default"] = so.intra_op_num_threads
        onx["inter_op_num_threads_default"] = so.inter_op_num_threads
        print(
            f"\nonnxruntime 默认线程: intra_op_num_threads={so.intra_op_num_threads}, inter_op_num_threads={so.inter_op_num_threads}"
            f"（0 = 用全部核；本机 {env_info.get('affinity_cpus')} 核可用）。"
            "每份文档的会话都默认占满全部核，MAX_PARALLEL_DOCUMENTS 份文档并发时会互相争抢，并发收益远低于份数。"
        )
except Exception as exc:  # noqa: BLE001
    unavailable(err(exc))
R["onnx"] = onx

## 7. Embedding 延迟

与 `folder_pipeline._preflight` 同一套工厂；3 次 × 16 条批请求。

In [ ]:
section("7. Embedding 延迟")
emb: dict = {}
try:
    from ragspine.common.evidence.providers.local_models import LocalEmbeddingAdapter
    from ragspine.common.evidence.providers.providers import load_local_model_config

    # 与 folder_pipeline._preflight 同一套工厂；未配置会抛 ValueError
    embedder = LocalEmbeddingAdapter(load_local_model_config("embedding"))
    base = "Quarterly revenue grew while operating margin held steady across the regional segments, according to the filing. "
    secs = []
    for call in range(3):
        texts = [f"[{call}-{j}-{time.time_ns()}] " + (base * 3)[:200] for j in range(16)]
        before = embedder.request_count
        t0 = time.perf_counter()
        vectors = embedder.embed_descriptions(texts)
        secs.append(time.perf_counter() - t0)
        emb.setdefault("requests_per_call", []).append(embedder.request_count - before)
        emb["dim"] = len(vectors[0]) if vectors else None
    avg = sum(secs) / len(secs)
    emb.update(call_seconds=[round(s, 3) for s in secs], avg_batch16_s=round(avg, 3))
    print_table(("第几次", "秒(16条)", "实际请求数"), [(i + 1, round(s, 3), emb["requests_per_call"][i]) for i, s in enumerate(secs)])
    print(f"平均 {avg:.3f}s / 16 条 (向量维度 {emb['dim']})")
    VECTORS_PER_PAGE = 10  # 假设
    pages = R["pdfspine"].get("pages")
    n_docs = R.get("pdf_count")
    if pages:
        vec = VECTORS_PER_PAGE * pages
        per_doc = vec / 16 * avg
        emb.update(assumed_vectors_per_page=VECTORS_PER_PAGE, vectors_per_doc=vec, doc_seconds=round(per_doc, 1))
        print(f"\n[假设] 每页 {VECTORS_PER_PAGE} 个向量 => 样例文档 {pages} 页 ≈ {vec} 个向量 => 约 {per_doc:.1f} s/份")
        if n_docs:
            emb.update(n_docs=n_docs, global_serial_total_s=round(per_doc * n_docs, 1))
            print(f"[假设] NB_PDF_DIR 共 {n_docs} 份 PDF，全局串行锁下 embedding 总时长 ≈ {per_doc * n_docs:.0f} s ({per_doc * n_docs / 60:.1f} 分钟)")
    else:
        print("样例 PDF 页数未知(第 5 节失败)，无法外推每份文档")
except Exception as exc:  # noqa: BLE001
    unavailable(err(exc))
R["embedding"] = emb

## 8. LLM 延迟

`RUN_LLM_PROBE`（或环境变量 `DIAG_RUN_LLM_PROBE=0`）控制；会产生 1～2 次计费调用。

In [ ]:
section("8. LLM 延迟")
# 会产生 1～2 次计费调用(1 次文本 JSON + 1 次 vision)；不想计费就改成 False，或运行前设环境变量 DIAG_RUN_LLM_PROBE=0
RUN_LLM_PROBE = os.environ.get("DIAG_RUN_LLM_PROBE", "1").strip().lower() not in ("0", "false", "no", "")
llm: dict = {}
try:
    if not RUN_LLM_PROBE:
        raise RuntimeError("RUN_LLM_PROBE = False，跳过")
    from enterprise_pdf_rag.adapters.http.layout_schemas import PageLayoutDTO
    from enterprise_pdf_rag.adapters.http.schemas import BoundaryModel
    from ragspine.common.evidence.providers.json_completion import JsonCompletionClient
    from ragspine.common.evidence.providers.providers import load_llm_config

    config = load_llm_config()  # 未配置抛 ValueError
    _diag_tmp = Path("/tmp/ragspine-diag")
    _diag_tmp.mkdir(parents=True, exist_ok=True)
    cache_dir = Path(tempfile.mkdtemp(prefix="llm-probe-", dir=_diag_tmp))
    try:
        client = JsonCompletionClient(config, cache_dir=cache_dir, max_live_calls=2, timeout=120.0)

        class _Pong(BoundaryModel):
            ok: bool

        t0 = time.perf_counter()
        client.complete_text_json(task="diag-text-probe", prompt=f"Reply with ok=true. nonce={time.time_ns()}", response_model=_Pong, max_output_tokens=64)
        llm["text_s"] = round(time.perf_counter() - t0, 2)
        print(f"文本 JSON 请求: {llm['text_s']} s")

        if SAMPLE_PDF is None:
            print("vision: UNAVAILABLE（没有样例 PDF）；只测了文本，表里按 文本×3 估")
        else:
            try:
                from enterprise_pdf_rag.adapters.figure_reasoning import render_svg_png
                from enterprise_pdf_rag.adapters.pdf_password import open_pdf

                d = open_pdf(SAMPLE_PDF.read_bytes())
                page = d.load_page(0)
                svg = page.get_svg_image(text_as_path=False)
                observations = []
                for block in page.get_text("dict").get("blocks", [])[:40]:
                    for line in block.get("lines", []):
                        for span in line.get("spans", []):
                            observations.append({"id": f"s{len(observations):04d}", "text": span.get("text", ""), "bbox": span.get("bbox")})
                d.close()
                t0 = time.perf_counter()
                png = render_svg_png(svg.encode() if isinstance(svg, str) else svg).png  # 与 page_partition._render 同一条路径
                llm["render_s"] = round(time.perf_counter() - t0, 2)
                prompt = (
                    "Map this page into layout regions (diagnostic probe, shorter than the production prompt). "
                    "Every source span alias must be assigned to a region or listed in unassigned_span_ids. "
                    f"nonce={time.time_ns()}\nSource text observations:\n" + json.dumps(observations, ensure_ascii=False, separators=(",", ":"))
                )
                t0 = time.perf_counter()
                client.complete_json(task="diag-vision-probe", prompt=prompt, image_png=png, response_model=PageLayoutDTO, max_output_tokens=4096)
                llm["vision_s"] = round(time.perf_counter() - t0, 2)
                print(f"vision 请求(page_partition 同一条路径: SVG→resvg PNG→complete_json): {llm['vision_s']} s，PNG 渲染 {llm['render_s']} s，PNG {len(png) // 1024} KB")
            except Exception as exc:  # noqa: BLE001
                print("vision UNAVAILABLE:", err(exc), "；只测了文本，表里按 文本×3 估")
    finally:
        shutil.rmtree(cache_dir, ignore_errors=True)
        try:
            _diag_tmp.rmdir()
        except OSError:
            pass
except Exception as exc:  # noqa: BLE001
    unavailable(err(exc))
R["llm"] = llm

## 9. 正在运行的 ingest 的线程栈采样

`py-spy dump` 3 次 × 间隔 15 秒；不可用时打印一段后台采样线程代码，放进 `run_folder.ipynb` 主运行格之前运行。

In [ ]:
section("9. 正在运行的 ingest 的线程栈采样")
BUCKETS = {
    "onnx": r"onnxruntime|find_layout",
    "fs": r"fsync|os\.replace|link_new_file|replace_file|read_bytes|scandir|\bstat\b",
    "net": r"socket|ssl|http\.client|recv",
    "lock": r"Lock\.acquire|Semaphore|Condition\.wait|acquire|\bwait\b",
    "render": r"resvg|render|ElementTree|crop_native_svg",
    "pydantic": r"pydantic|model_validate|TypeAdapter",
    "embed": r"embed|embedding",
}
SAMPLES, INTERVAL = 3, 15


def bucketize(text: str) -> Counter:
    counts: Counter = Counter()
    for line in text.splitlines():
        for name, pattern in BUCKETS.items():
            if re.search(pattern, line, re.IGNORECASE):
                counts[name] += 1
    return counts


FALLBACK_SNIPPET = '''\
# 把这一格放在 run_folder.ipynb 里"主运行"那一格的【前面】并先运行它(Databricks 同一 kernel 的格子是排队执行的，
# ingest 开始后就没法再插一格运行了)。它只启动一个后台守护线程：等 DELAY_S 秒(让 ingest 进入稳定阶段)后，
# 每 15 秒用 sys._current_frames() 抓一次本进程所有线程的栈，共 3 次，做分桶计数，写到文件并打印。
# 采样结果文件: /tmp/ragspine-diag/stack-samples.txt (ingest 结束后 print(open(path).read()) 即可，发回这个文件内容)
import re, sys, threading, time, traceback
from collections import Counter
from pathlib import Path

DELAY_S, INTERVAL_S, ROUNDS = 120, 15, 3
OUT = Path("/tmp/ragspine-diag/stack-samples.txt")
BUCKETS = {
    "onnx": r"onnxruntime|find_layout",
    "fs": r"fsync|os\\.replace|link_new_file|replace_file|read_bytes|scandir|\\bstat\\b",
    "net": r"socket|ssl|http\\.client|recv",
    "lock": r"Lock\\.acquire|Semaphore|Condition\\.wait|acquire|\\bwait\\b",
    "render": r"resvg|render|ElementTree|crop_native_svg",
    "pydantic": r"pydantic|model_validate|TypeAdapter",
    "embed": r"embed|embedding",
}


def _sample_stacks():
    OUT.parent.mkdir(parents=True, exist_ok=True)
    me = threading.get_ident()
    total = Counter()
    with OUT.open("w", encoding="utf-8") as out:
        time.sleep(DELAY_S)
        for round_no in range(ROUNDS):
            names = {t.ident: t.name for t in threading.enumerate()}
            counts = Counter()
            print("=" * 60, "sample", round_no + 1, time.strftime("%H:%M:%S"), file=out)
            for ident, frame in sys._current_frames().items():
                if ident == me:
                    continue
                stack = "".join(traceback.format_stack(frame))
                print("--- thread", ident, names.get(ident, "?"), file=out)
                print("".join(stack.splitlines(True)[-12:]), file=out)
                for line in stack.splitlines():
                    for name, pattern in BUCKETS.items():
                        if re.search(pattern, line, re.IGNORECASE):
                            counts[name] += 1
            print("bucket counts:", dict(counts), file=out, flush=True)
            total.update(counts)
            if round_no < ROUNDS - 1:
                time.sleep(INTERVAL_S)
        print("TOTAL buckets:", dict(total.most_common()), file=out, flush=True)


threading.Thread(target=_sample_stacks, name="stack-sampler", daemon=True).start()
print(f"已启动后台采样线程：{DELAY_S}s 后开始，结果写入 {OUT}")
'''

stack: dict = {"method": None, "buckets": {}, "targets": []}
try:
    if os.name == "nt":
        raise RuntimeError("Windows 没有 ps")
    out = subprocess.run(["ps", "-eo", "pid,etime,args"], capture_output=True, text=True, timeout=20, check=True).stdout
    cands = []
    for line in out.splitlines()[1:]:
        parts = line.strip().split(None, 2)
        if len(parts) < 3 or not parts[0].isdigit():
            continue
        pid, etime, args = int(parts[0]), parts[1], parts[2]
        if pid == os.getpid() or "ps -eo" in args or "py-spy" in args:
            continue
        if re.search(r"ipykernel|run_folder|folder_pipeline|python", args, re.IGNORECASE):
            cands.append((pid, etime, args))
    cands.sort(key=lambda c: 0 if re.search(r"ipykernel|run_folder|folder_pipeline", c[2], re.IGNORECASE) else 1)
    print(f"候选 python 进程 {len(cands)} 个(本进程 PID {os.getpid()} 已排除):")
    print_table(("PID", "etime", "cmdline(前 100 字符)"), [(p, e, a[:100]) for p, e, a in cands] or [("-", "-", "(无)")])
    stack["targets"] = [c[0] for c in cands]
except Exception as exc:  # noqa: BLE001
    cands = []
    unavailable(f"列进程失败: {err(exc)}")

pyspy = None
try:
    if cands:
        _bin = Path(sys.executable).parent / "py-spy"
        pyspy = shutil.which("py-spy") or (str(_bin) if _bin.exists() else None)
        if pyspy is None:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "py-spy"], capture_output=True, text=True, timeout=240)
            pyspy = shutil.which("py-spy") or (str(_bin) if _bin.exists() else None)
        if pyspy is None:
            raise RuntimeError("py-spy 安装失败或不在 PATH 上")
        ver = subprocess.run([pyspy, "--version"], capture_output=True, text=True, timeout=20)
        print("py-spy:", pyspy, ver.stdout.strip())
except Exception as exc:  # noqa: BLE001
    pyspy = None
    unavailable(f"py-spy: {err(exc)}")

attached = 0
try:
    if pyspy and cands:
        targets = cands[:8]
        if len(cands) > 8:
            print(f"候选超过 8 个，只采样前 8 个(优先 ipykernel / run_folder / folder_pipeline)")
        total: Counter = Counter()
        first_dump: dict[int, str] = {}
        failures: dict[int, str] = {}
        for sample in range(SAMPLES):
            for pid, _etime, _args in targets:
                if pid in failures:
                    continue
                res = subprocess.run([pyspy, "dump", "--pid", str(pid), "--nonblocking"], capture_output=True, text=True, timeout=30)
                if res.returncode != 0:
                    failures[pid] = (res.stderr.strip().splitlines() or ["exit " + str(res.returncode)])[-1]
                    continue
                first_dump.setdefault(pid, res.stdout)
                total.update(bucketize(res.stdout))
            if sample < SAMPLES - 1:
                time.sleep(INTERVAL)
        attached = len(first_dump)
        for pid, why in failures.items():
            print(f"PID {pid}: UNAVAILABLE: py-spy dump 失败: {why}")
        if attached:
            stack["method"] = "py-spy"
            stack["buckets"] = dict(total.most_common())
            print(f"\n成功 attach {attached} 个进程，{SAMPLES} 次采样 × 间隔 {INTERVAL}s。分桶计数(所有进程、所有线程、所有采样的栈行数):")
            print_table(("桶", "正则", "计数"), [(k, BUCKETS[k], total.get(k, 0)) for k in BUCKETS])
            for pid, dump in first_dump.items():
                print(f"\n--- PID {pid} 第一次采样原始输出前 80 行 ---")
                print("\n".join(dump.splitlines()[:80]))
except Exception as exc:  # noqa: BLE001
    unavailable(f"采样失败: {err(exc)}")

if not attached:
    print("\npy-spy 不可用或没有权限 attach(Databricks 上常见: ptrace 被禁)。改用下面这段代码:")
    print("把它放进 run_folder.ipynb 主运行格的前面先运行(后台线程在 ingest 同一进程内用 sys._current_frames() 采样，不需要权限)，ingest 跑过约 3 分钟后把 /tmp/ragspine-diag/stack-samples.txt 的内容发回。\n")
    print(FALLBACK_SNIPPET)
R["stack"] = stack

## 10. 汇总判断

汇成一张表并给出结论，写出 JSON / MD 报告到 `ROOT_DIR/data/_diagnostics/`。

In [ ]:
section("10. 汇总判断")
sec = lambda x: f"{x:.0f}" if x >= 100 else (f"{x:.1f}" if x >= 10 else f"{x:.2f}")  # noqa: E731
rows: list[dict] = []
notes: list[str] = []
try:
    pages = R["pdfspine"].get("pages")
    ex = R["pdfspine"]
    FALLBACK_RATIO, FILES_PER_PAGE, VERIFY_REREADS = 0.20, 80, 3  # 假设
    # 1 pdfspine 抽取
    rows.append(dict(item="pdfspine 抽取", seconds=ex.get("doc_seconds"), basis="实测/外推(实测 ms/页 × 页数)", amortize="部分: 多文档线程并发，取决于 pdfspine 是否释放 GIL"))
    # 2 ONNX
    if R["onnx"].get("usable"):
        c = R["onnx"].get("concurrency", {})
        rows.append(dict(item="ONNX 版面", seconds=R["onnx"]["doc_minutes"] * 60, basis="实测/外推(实测 s/页 × 页数)", amortize=f"有限: ort 默认占满全部核，并发实测加速比 {c.get('speedup', '未测')}x"))
    else:
        rows.append(dict(item="ONNX 版面", seconds=0.0, basis="不可用 → run_folder 走 deterministic-text-pages", amortize="-"))
    # 3 回退页 LLM
    unit = R["llm"].get("vision_s") or (R["llm"].get("text_s") * 3 if R["llm"].get("text_s") else None)
    unit_src = "vision 实测" if R["llm"].get("vision_s") else "文本实测×3"
    llm_s = pages * FALLBACK_RATIO * unit if (pages and unit) else None
    rows.append(dict(item="回退页 LLM", seconds=llm_s, basis=f"假设(回退页 {FALLBACK_RATIO:.0%} × 页数) × {unit_src} 单价" if unit else "UNAVAILABLE(第 8 节无数据)", amortize="能: 最多 MAX_PARALLEL_DOCUMENTS 路，受端点限速"))
    # 4 embedding
    rows.append(dict(item="embedding(全局串行)", seconds=R["embedding"].get("doc_seconds"), basis="外推 + 假设(每页 10 个向量)" if R["embedding"].get("doc_seconds") is not None else "UNAVAILABLE(第 7 节无数据)", amortize="否: 全局串行锁，并发后成为天花板"))
    # 5 文件系统
    fs = R["fsbench"].get("ROOT_DIR/data")
    if fs and pages:
        files = pages * FILES_PER_PAGE
        fs_s = files * (fs["repo_ms"] + VERIFY_REREADS * (fs["stat_ms"] + fs["read_ms"])) / 1000
        rows.append(dict(item="文件系统(落位 + 校验重读)", seconds=fs_s, basis=f"假设(每页 {FILES_PER_PAGE} 个文件, 校验重读 {VERIFY_REREADS} 次) × 实测 ROOT_DIR/data 单价", amortize="部分: FUSE 往返可被多文档并行重叠，但共享同一挂载"))
    else:
        rows.append(dict(item="文件系统(落位 + 校验重读)", seconds=None, basis="UNAVAILABLE(第 4/5 节无数据)", amortize="部分"))
    rows.sort(key=lambda r: (r["seconds"] is None, -(r["seconds"] or 0)))
    print_table(
        ("耗时项", "每份文档估计秒数", "依据", "能否被并发摊薄"),
        [(r["item"], "n/a" if r["seconds"] is None else sec(r["seconds"]), r["basis"], r["amortize"]) for r in rows],
    )
    ranked = [r for r in rows if r["seconds"]]
    if len(ranked) >= 2:
        a, b = ranked[0], ranked[1]
        ratio = a["seconds"] / b["seconds"] if b["seconds"] else float("inf")
        notes.append(f"第一名是「{a['item']}」，约 {sec(a['seconds'])} 秒/份；是第二名「{b['item']}」({sec(b['seconds'])} 秒/份) 的 {ratio:.1f} 倍。")
    elif ranked:
        notes.append(f"只有「{ranked[0]['item']}」有数据({sec(ranked[0]['seconds'])} 秒/份)，其余项见上表 n/a，无法比较。")
    else:
        notes.append("没有任何一项有可比数据(见上方各节 UNAVAILABLE)。")
    assumed = ["每页向量数 10(embedding)", f"每页文件数 {FILES_PER_PAGE} 与校验重读 {VERIFY_REREADS} 次(文件系统)", f"回退页比例 {FALLBACK_RATIO:.0%}(回退页 LLM)"]
    notes.append("下列数字是假设，需要用第 9 节栈采样验证: " + "；".join(assumed) + "。")
    if R["progress"].get("files_per_doc_measured") and pages:
        notes.append(f"交叉检查: 第 3 节实测已完成文档 processing/ 平均 {R['progress']['files_per_doc_measured']} 个文件(样例文档 {pages} 页，假设值 {pages * FILES_PER_PAGE} 个，两者不同页数不可直接比，仅作数量级参考)。")
    # 栈采样 vs 表格
    bucket_to_item = {"onnx": "ONNX 版面", "fs": "文件系统(落位 + 校验重读)", "net": "回退页 LLM 或 embedding(网络等待)", "lock": "embedding(全局串行)或其它锁等待", "render": "回退页 LLM(渲染)", "pydantic": "pdfspine 抽取(校验)", "embed": "embedding(全局串行)"}
    sb = R["stack"].get("buckets") or {}
    if sb and sum(sb.values()):
        top = max(sb, key=sb.get)
        agree = ranked and (bucket_to_item[top].startswith(ranked[0]["item"][:6]) or ranked[0]["item"].startswith(bucket_to_item[top][:6]))
        notes.append(f"栈采样最多的桶是「{top}」({sb[top]} 行)，对应「{bucket_to_item[top]}」；表格第一名是「{ranked[0]['item'] if ranked else 'n/a'}」: {'一致，结论较可靠' if agree else '不一致，以栈采样为准，回头检查上面的假设'}。")
        notes.append("注意: 空闲 worker 线程(等任务)也会命中 lock 桶的 wait，lock 计数偏高时先看其它桶。")
    else:
        notes.append("第 9 节没有栈采样结果: 请把它打印的 sys._current_frames 后台采样代码放进 run_folder.ipynb 主运行格之前运行，并把 /tmp/ragspine-diag/stack-samples.txt 发回，用来验证上述假设。")
    print()
    for n in notes:
        print("-", n)
except Exception as exc:  # noqa: BLE001
    unavailable(err(exc))
    notes.append(f"汇总失败: {err(exc)}")

# 写 JSON + MD
try:
    DIAG_DIR.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
    report = dict(generated_utc=stamp, table=rows, conclusion=notes, **R)
    json_path = DIAG_DIR / f"ingest_diagnostics_{stamp}.json"
    json_path.write_text(json.dumps(report, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    md = [f"# ingest 诊断 {stamp}", "", "## 汇总表", "", "| 耗时项 | 每份文档估计秒数 | 依据 | 能否被并发摊薄 |", "|---|---|---|---|"]
    md += [f"| {r['item']} | {'n/a' if r['seconds'] is None else sec(r['seconds'])} | {r['basis']} | {r['amortize']} |" for r in rows]
    md += ["", "## 结论", ""] + [f"- {n}" for n in notes]
    md += ["", "## 环境", ""] + [f"- {k}: {v}" for k, v in R.get("env", {}).items()]
    md_path = json_path.with_suffix(".md")
    md_path.write_text("\n".join(md) + "\n", encoding="utf-8")
    print("\n报告已写入(请把这两个文件发回):\n ", json_path, "\n ", md_path)
except Exception as exc:  # noqa: BLE001
    unavailable(f"写报告失败: {err(exc)}")